# Panic analysis implementation: grounding before code changes

Evaluate four proposed implementation areas against fresh source controls, bounded mathematical models and compiler evidence. Research only; production extraction and solver code are unchanged.

1. Lexical recursion and receiver binding: positive and adversarial controls.
2. Source, executable body and invocation identities: keep distinct; values need typed origins and evaluation guards.
3. SAT supplies an encoded witness to replay. UNSAT excludes counterexamples only in the complete declared domain and exploration bound. Missing facts or an open frontier remain inconclusive.
4. Traits, expansions and implicit checks retain compiler validation. The known CGA virtual-call substitution defect is not assumed fixed.

Local evidence: `docs/research/artifacts/2026-11-01-panic-implementation-grounding/`. Prior evaluation: `docs/research/artifacts/2026-11-01-spur-math-panic/README.md`. Tracker bd-39y; independent source/compiler lane bd-3dw.

Sources: [Rust method lookup](https://doc.rust-lang.org/reference/expressions/method-call-expr.html), [lexical scopes](https://doc.rust-lang.org/reference/names/scopes.html), [operators and checks](https://doc.rust-lang.org/reference/expressions/operator-expr.html), [CGA approach](https://tweedegolf.nl/en/blog/242/finding-panics-with-call-graph-analyzer/).

In [2]:
from pathlib import Path
import sys, json, hashlib, subprocess, importlib.util, platform
pg_repo = Path("/Users/kevintruong/project/SpurCode")
pg_out = pg_repo / "docs/research/artifacts/2026-11-01-panic-implementation-grounding"
pg_sem = pg_out / "semantics"
pg_out.mkdir(parents=True, exist_ok=True)
pg_protected_names = ["crates/spur-acp/src/adapter/codex.rs", "crates/spur-acp/src/adapter/mod.rs", "docs/critical-findings.md", "docs/research/2026-10-27-beads-maintenance-analysis.ipynb", "duckdb-extension-journey-interactive-20261027.ipynb", "2026-10-31-panic-graph-experiment.ipynb"]
def pg_sha(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()
pg_current = pg_repo / ".spur/graph/CURRENT"
pg_before = {"head": subprocess.check_output(["git","rev-parse","HEAD"],cwd=pg_repo,text=True).strip(),
             "protected": {p: pg_sha(pg_repo/p) for p in pg_protected_names},
             "graph_current_bytes": pg_current.read_text() if pg_current.is_file() else None,
             "python": sys.version, "platform": platform.platform()}
(pg_out/"before.json").write_text(json.dumps(pg_before,indent=2)+"\n")
pg_spec = importlib.util.spec_from_file_location("pg_probe", pg_sem/"probe.py")
pg_probe = importlib.util.module_from_spec(pg_spec)
pg_spec.loader.exec_module(pg_probe)
pg_setup_result = {"build":pg_probe.build(),"finite":pg_probe.verify_finite()}
import duckdb
pg_conn=duckdb.connect()
pg_extensions = pg_conn.execute("select extension_name,installed,loaded,extension_version from duckdb_extensions() where extension_name in ('json','parquet','petgraph_ext','duckrouting','shellfs') order by extension_name").fetchall()
pg_probe.dump(pg_out/"extensions.json",{"duckdb":duckdb.__version__,"extensions":pg_extensions})
print(json.dumps({"setup":pg_setup_result,"duckdb":duckdb.__version__,"extensions":pg_extensions,"CURRENT":pg_before["graph_current_bytes"],"protected":len(pg_before["protected"])},indent=2))

{
  "setup": {
    "build": {
      "typed_cases": 19,
      "source_functions": 16,
      "source_sha256": "8cdea6c59781e165b66eb063be444cf80452b8fba35128456964e6f19d61263f"
    },
    "finite": {
      "assignments": 15872,
      "cases": 19,
      "passed": true
    }
  },
  "duckdb": "1.5.5",
  "extensions": [
    [
      "duckrouting",
      true,
      false,
      "509ece1"
    ],
    [
      "json",
      true,
      true,
      "v1.5.5"
    ],
    [
      "parquet",
      true,
      true,
      "v1.5.5"
    ],
    [
      "petgraph_ext",
      true,
      false,
      "v0.5.0"
    ],
    [
      "shellfs",
      true,
      false,
      "4f724fc"
    ]
  ],
  "CURRENT": null,
  "protected": 6
}


In [5]:
import os, time
pg_runtime_commands = []
for pg_profile, pg_checks in [("checks_on","on"),("checks_off","off")]:
    pg_env = os.environ.copy()
    pg_env.update({"SPUR_REMOTE":"0","RUSTUP_TOOLCHAIN":"nightly-2026-10-18","CARGO_TARGET_DIR":f"/private/tmp/spur-panic-grounding-{pg_profile}-target","RUSTFLAGS":f"-Cpanic=unwind -Cdebug-assertions=on -Coverflow-checks={pg_checks}"})
    pg_cmd = [str(pg_repo/"scripts/spur-cargo"),"run","--offline","--manifest-path",str(pg_sem/"runtime/Cargo.toml"),"--bin","spur-panic-grounding-control","--",pg_profile]
    pg_started=time.monotonic()
    with (pg_sem/f"runtime-{pg_profile}.jsonl").open("w") as pg_stdout, (pg_sem/f"runtime-{pg_profile}.stderr").open("w") as pg_stderr:
        pg_ret=subprocess.run(pg_cmd,cwd=pg_repo,env=pg_env,stdout=pg_stdout,stderr=pg_stderr,timeout=60)
    pg_binary=Path(pg_env["CARGO_TARGET_DIR"])/"aarch64-apple-darwin/debug/spur-panic-grounding-control"
    if not pg_binary.is_file():
        pg_candidates=list(Path(pg_env["CARGO_TARGET_DIR"]).rglob("debug/spur-panic-grounding-control"))
        assert len(pg_candidates)==1,pg_candidates
        pg_binary=pg_candidates[0]
    pg_runtime_commands.append({"profile":pg_profile,"command":pg_cmd,"exit_code":pg_ret.returncode,"seconds":time.monotonic()-pg_started,"env":{k:pg_env[k] for k in ["SPUR_REMOTE","RUSTUP_TOOLCHAIN","CARGO_TARGET_DIR","RUSTFLAGS"]},"binary":str(pg_binary),"binary_sha256":pg_sha(pg_binary)})
    assert pg_ret.returncode==0,(pg_profile,(pg_sem/f"runtime-{pg_profile}.stderr").read_text()[-3000:])
pg_probe.dump(pg_sem/"runtime-commands.json",pg_runtime_commands)
pg_rustc=subprocess.run(["rustup","run","nightly-2026-10-18","rustc","-Vv"],capture_output=True,text=True,check=True)
(pg_sem/"runtime-rustc.txt").write_text(pg_rustc.stdout)
pg_runtime_result=pg_probe.verify_runtime()
pg_prior_inputs=json.loads((pg_repo/"docs/research/artifacts/2026-11-01-execution-cga-crosscheck/inputs.json").read_text())
pg_graph_protection={"CURRENT":str(pg_current.resolve()),"is_symlink":pg_current.is_symlink(),"inputs":[]}
for pg_input in [r for r in pg_prior_inputs["inputs"] if "/.spur/graph/" in r["path"] and "sha256" in r]:
    pg_graph_protection["inputs"].append({"path":pg_input["path"],"sha256":pg_sha(pg_input["path"]),"matches_prior":pg_sha(pg_input["path"])==pg_input["sha256"]})
assert all(x["matches_prior"] for x in pg_graph_protection["inputs"])
pg_probe.dump(pg_out/"graph-protection-before.json",pg_graph_protection)
print(json.dumps({"runtime_executions":pg_runtime_result["executions"],"passed":pg_runtime_result["passed"],"commands":[{k:r[k] for k in ["profile","exit_code","seconds","binary_sha256"]} for r in pg_runtime_commands],"frozen_graph_tables_unchanged":len(pg_graph_protection["inputs"]),"rustc":pg_rustc.stdout},indent=2))

{
  "runtime_executions": 7680,
  "passed": true,
  "commands": [
    {
      "profile": "checks_on",
      "exit_code": 0,
      "seconds": 0.6782768749981187,
      "binary_sha256": "f72d11ef4c72056c918ccefb6f52747a209fe7b483a57f68862ca8b5dc426cf8"
    },
    {
      "profile": "checks_off",
      "exit_code": 0,
      "seconds": 0.6177434579876717,
      "binary_sha256": "29840efca171fd84348832f07c62af775d6887bf8fee1215e67ae0d29463bf53"
    }
  ],
  "frozen_graph_tables_unchanged": 7,
  "rustc": "rustc 1.101.0-nightly (8006091eb 2026-10-17)\nbinary: rustc\ncommit-hash: 8006091eb71ae7d9e9bccb66c6122c9d68af4466\ncommit-date: 2026-10-17\nhost: aarch64-apple-darwin\nrelease: 1.101.0-nightly\nLLVM version: 23.1.4\n"
}


In [6]:
pg_conn.execute("LOAD petgraph_ext")
pg_conn.execute("LOAD duckrouting")
pg_ext_functions=pg_conn.execute("select function_name,function_type,parameters,parameter_types,return_type from duckdb_functions() where function_name like 'pg_%' or function_name like 'duckrouting%' order by function_name").fetchall()
pg_probe.dump(pg_out/"extension-functions.json",pg_ext_functions)
print(json.dumps({"functions":pg_ext_functions},default=str,indent=2))

{
  "functions": [
    [
      "duckrouting_articulation_points",
      "table",
      [
        "col0"
      ],
      [
        "VARCHAR"
      ],
      null
    ],
    [
      "duckrouting_astar",
      "table",
      [
        "col0",
        "col1",
        "col2",
        "col3",
        "directed",
        "heuristic",
        "factor",
        "epsilon"
      ],
      [
        "VARCHAR",
        "BIGINT[]",
        "BIGINT[]",
        "BOOLEAN",
        "BOOLEAN",
        "BIGINT",
        "DOUBLE",
        "DOUBLE"
      ],
      null
    ],
    [
      "duckrouting_astar",
      "table",
      [
        "col0",
        "col1",
        "col2",
        "directed",
        "heuristic",
        "factor",
        "epsilon"
      ],
      [
        "VARCHAR",
        "BIGINT[]",
        "BIGINT[]",
        "BOOLEAN",
        "BIGINT",
        "DOUBLE",
        "DOUBLE"
      ],
      null
    ],
    [
      "duckrouting_astar",
      "table",
      [
        "col0",
        "col1",

In [7]:
pg_fn_signatures=pg_conn.execute("select function_name,function_type,parameters,parameter_types,return_type from duckdb_functions() where function_name in ('petgraph_shortest_path','duckrouting_dijkstra','petgraph_strongly_connected_components','duckrouting_strong_components') order by function_name").fetchall()
pg_probe.dump(pg_out/"extension-selected-signatures.json",pg_fn_signatures)
print(json.dumps(pg_fn_signatures,indent=2))

[
  [
    "duckrouting_dijkstra",
    "table",
    [
      "col0",
      "col1",
      "col2",
      "directed"
    ],
    [
      "VARCHAR",
      "BIGINT",
      "BIGINT",
      "BOOLEAN"
    ],
    null
  ],
  [
    "duckrouting_dijkstra",
    "table",
    [
      "col0",
      "col1",
      "col2",
      "col3",
      "directed"
    ],
    [
      "VARCHAR",
      "BIGINT",
      "BIGINT",
      "BOOLEAN",
      "BOOLEAN"
    ],
    null
  ],
  [
    "duckrouting_dijkstra",
    "table",
    [
      "col0",
      "col1",
      "col2",
      "directed"
    ],
    [
      "VARCHAR",
      "BIGINT",
      "BIGINT[]",
      "BOOLEAN"
    ],
    null
  ],
  [
    "duckrouting_dijkstra",
    "table",
    [
      "col0",
      "col1",
      "col2",
      "col3",
      "directed"
    ],
    [
      "VARCHAR",
      "BIGINT",
      "BIGINT[]",
      "BOOLEAN",
      "BOOLEAN"
    ],
    null
  ],
  [
    "duckrouting_dijkstra",
    "table",
    [
      "col0",
      "col1",
      "col2",
   

In [8]:
sys.path.insert(0,str(pg_sem))
import verify_evidence as pg_verify, replay_smt as pg_smt
pg_solver_verified=pg_verify.verify()
pg_smt_result=pg_smt.run()
pg_finite_result=pg_probe.verify_finite()
print(json.dumps({"receipt_and_runtime_verification":pg_solver_verified,"fresh_z3_formula_replay":pg_smt_result,"independent_finite":pg_finite_result},indent=2))

{
  "receipt_and_runtime_verification": {
    "passed": true,
    "source_sites": 17,
    "typed_solves": 19,
    "catalog_solves": 9,
    "reloaded_solves": 28,
    "cores_checked": 9,
    "complete_models_runtime_matched": 9,
    "spurious_missing_origin_witness_rejected": true,
    "semantic_facts": "author-normalized for the bounded fixture only"
  },
  "fresh_z3_formula_replay": {
    "passed": true,
    "z3": "/opt/homebrew/bin/z3",
    "z3_version": "Z3 version 4.16.0 - 64 bit",
    "executions": 28,
    "sat": 14,
    "unsat": 14,
    "unknown_or_timeout": 0,
    "handwritten_assertions_added": false,
    "original_formula_prefixes_preserved": true
  },
  "independent_finite": {
    "assignments": 15872,
    "cases": 19,
    "passed": true
  }
}


In [10]:
pg_snapshot_exe=Path("/private/tmp/panic-workspace-20261101/target/aarch64-apple-darwin/debug/examples/panic_evidence")
assert pg_sha(pg_snapshot_exe)=="9d515d0b0160afef11b9063e24383f621006332be14cba6db9259899a5d46777"
import tempfile
pg_snapshot_root=Path(tempfile.mkdtemp(prefix="spur-panic-grounding-source-"))
(pg_snapshot_root/"lib.rs").write_bytes((pg_sem/"fixtures/lib.rs").read_bytes())
assert pg_sha(pg_snapshot_root/"lib.rs")==pg_sha(pg_sem/"fixtures/lib.rs")
pg_snapshot_request={"command":"snapshot","root":str(pg_snapshot_root),"source_paths":["lib.rs"]}
pg_snapshot_run=subprocess.run([str(pg_snapshot_exe)],input=json.dumps(pg_snapshot_request),capture_output=True,text=True,timeout=30)
pg_probe.dump(pg_sem/"snapshot-command.json",{"request":pg_snapshot_request,"command":[str(pg_snapshot_exe)],"binary_sha256":pg_sha(pg_snapshot_exe),"exit_code":pg_snapshot_run.returncode,"stderr":pg_snapshot_run.stderr,"freshness":"retained extractor executable; binary identity pinned, no production changes made"})
assert pg_snapshot_run.returncode==0,pg_snapshot_run.stderr
pg_snapshot=json.loads(pg_snapshot_run.stdout)
pg_probe.dump(pg_sem/"source-snapshot.json",pg_snapshot)
print(json.dumps({"top_keys":list(pg_snapshot),"sizes":{k:len(v) for k,v in pg_snapshot.items() if isinstance(v,(dict,list))},"preview":str(pg_snapshot)[:1600]},indent=2))

{
  "top_keys": [
    "graph",
    "method_sites"
  ],
  "sizes": {
    "graph": 9,
    "method_sites": 6
  },
  "preview": "{'graph': {'header': {'graph_index_version': '4'}, 'manifest_version': '588b0718b0ca5800de3507f603cb809ed90bfa932c63f04865902ef417c4e8c0', 'graph_content_hash': '5a0a224acd3ffa2092365b5eb5f204efdb78dc82f768f28ba25d113dce5958f2', 'file_manifests': [{'stable_file_id': '1892e6415d438ddc', 'path': 'lib.rs', 'content_oid': 'dc6c53db691876b2872ac747a7855d03a32e7c84', 'node_ids': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]}], 'files': [{'stable_file_id': '1892e6415d438ddc', 'file_path': 'lib.rs'}], 'symbols': [{'stable_symbol_id': '8a8396da569067fc', 'file_path': 'lib.rs', 'byte_range': [107, 228], 'line_range': [3, 3], 'entity_name': 'phi_safe', 'qualified_name': 'phi_safe', 'symbol_kind': 'function', 'anchor_hash': '3464040521443487252', 'enclosing_scope': None}, {'stable_symbol_id': 'b419b04711052a3c', 'file_path': 'lib.rs', 'byte_range': 

In [11]:
import source_grounding as pg_source, control_graph as pg_control
pg_source_grounded=pg_source.run()
pg_graph_control=pg_control.run(pg_conn)
print(json.dumps({"source_snapshot":{"body_ranges_matched":pg_source_grounded["body_ranges_matched"],"unwrap_sites":pg_source_grounded["unwrap_sites"],"wrong_receiver_sites":pg_source_grounded["wrong_receiver_sites"],"shadowed_method_sites":pg_source_grounded["shadowed_method_sites"],"shadowed_graph_pairs":pg_source_grounded["shadowed_graph_pairs"]},"extension_control":pg_graph_control},indent=2))

{
  "source_snapshot": {
    "body_ranges_matched": 16,
    "unwrap_sites": 12,
    "wrong_receiver_sites": 11,
    "shadowed_method_sites": 2,
    "shadowed_graph_pairs": 1
  },
  "extension_control": {
    "passed": true,
    "directed_checks": 4,
    "same_topology": true,
    "phi_safe": "structural path, UNSAT predicate, no runtime panics",
    "phi_bad": "same structural path, SAT predicate, 255 runtime panics per profile"
  }
}


In [12]:
pg_method_rows=pg_source_grounded["rows"]
pg_solver_rows=[json.loads(line) for line in (pg_sem/"solver-summary.jsonl").read_text().splitlines()]
pg_runtime_counts=pg_runtime_result["counts"]
pg_probe.dump(pg_sem/"runtime-summary.json",pg_runtime_counts)
pg_probe.dump(pg_sem/"source-binding-summary.json",[{k:r[k] for k in ["function","method_site_id","byte_range","receiver_text","expected_callee_from_typed_fixture","wrong_typed_receiver_binding"]} for r in pg_method_rows])
pg_schemas={}
for pg_name in ["solver-summary.jsonl","runtime-summary.json","source-binding-summary.json"]:
    pg_schemas[pg_name]=pg_conn.execute("DESCRIBE SELECT * FROM read_json_auto(?)",[str(pg_sem/pg_name)]).fetchall()
pg_probe.dump(pg_out/"notebook-table-schemas.json",pg_schemas)
print(json.dumps(pg_schemas,indent=2))

{
  "solver-summary.jsonl": [
    [
      "core",
      "VARCHAR[]",
      "YES",
      null,
      null,
      null
    ],
    [
      "function",
      "VARCHAR",
      "YES",
      null,
      null,
      null
    ],
    [
      "interpretation_gate",
      "VARCHAR",
      "YES",
      null,
      null,
      null
    ],
    [
      "model",
      "STRUCT(reached BOOLEAN, \"some\" BOOLEAN, x BIGINT, step BIGINT)",
      "YES",
      null,
      null,
      null
    ],
    [
      "name",
      "VARCHAR",
      "YES",
      null,
      null,
      null
    ],
    [
      "profile",
      "VARCHAR",
      "YES",
      null,
      null,
      null
    ],
    [
      "solve_id",
      "VARCHAR",
      "YES",
      null,
      null,
      null
    ],
    [
      "source_sha256",
      "VARCHAR",
      "YES",
      null,
      null,
      null
    ],
    [
      "status",
      "VARCHAR",
      "YES",
      null,
      null,
      null
    ],
    [
      "witness_replay_panic",
      "BO

## Solver results joined to exhaustive runtime outcomes

The exact fixture is compiled twice with `overflow-checks=on/off`, `panic=unwind`, and debug assertions on. Every `u8` input is exercised for 15 functions: 7,680 function executions. The 19 typed requests are preflighted, persisted and reloaded; nine additional catalog cases cover identity/presence and bounded workflow rules. All 28 returned formulas are freshly replayed with Z3 4.16.0.

The following table preserves raw solver status separately from the interpretation gate. A missing value fact produces a SAT witness that does not panic on replay. A stale origin and an open search frontier can produce UNSAT. These results do not establish source safety. Semantic formulas here are explicitly authored for the fixture and checked against Rust execution; this is not an automatic Rust semantic extractor.

In [13]:
SELECT s.name, s.profile, s.status, s.witness_x, s.witness_replay_panic,
       r.panics AS runtime_panics_of_256, s.interpretation_gate
FROM read_json_auto('docs/research/artifacts/2026-11-01-panic-implementation-grounding/semantics/solver-summary.jsonl') AS s
JOIN read_json_auto('docs/research/artifacts/2026-11-01-panic-implementation-grounding/semantics/runtime-summary.json') AS r
  ON s.function = r.function AND s.profile = r.profile
ORDER BY s.name;

name,profile,status,witness_x,witness_replay_panic,runtime_panics_of_256,interpretation_gate
add_checked,checks_on,sat,255,True,1,complete_declared_domain
add_guarded,checks_on,unsat,<NA>,None,0,complete_declared_domain
add_plain_off,checks_off,unsat,<NA>,None,0,complete_declared_domain
add_plain_on,checks_on,sat,255,True,1,complete_declared_domain
call_bad,checks_on,sat,1,True,255,complete_declared_domain
call_depth_zero,checks_on,unsat,<NA>,None,255,unknown_open_call_frontier
call_safe,checks_on,unsat,<NA>,None,0,complete_declared_domain
index_bad,checks_on,sat,3,True,253,complete_declared_domain
index_guarded,checks_on,unsat,<NA>,None,0,complete_declared_domain
loop_bound_1,checks_on,unsat,<NA>,None,1,unknown_open_loop_frontier


## Current source evidence: receiver identity fails before satisfiability

The retained hash-pinned extractor ran on the same fresh source bytes used for the runtime controls. All 16 ordinary function ranges match the author-bound ranges. The method sidecar retains 12 distinct `unwrap` invocation spans, including both shadowed `value.unwrap()` calls. The ordinary graph collapses those two calls into one caller/callee pair.

Eleven Option receiver sites resolve to the in-scope custom `Lookalike::unwrap`, whose actual method returns 7 without panicking. Those receivers are Options in the compiled fixture; a matching method name is insufficient. The method sidecar correctly keeps `compiler_target=unknown`; its exact source span does not certify the graph edge's target.

This is a fresh fixture run with the retained extractor executable, not a newly rebuilt application or a published graph. Independent current-source/freshness checks and lexical/compiler controls are evaluated in the grounding lane.

In [14]:
SELECT function, byte_range, receiver_text, expected_callee_from_typed_fixture,
       wrong_typed_receiver_binding
FROM read_json_auto('docs/research/artifacts/2026-11-01-panic-implementation-grounding/semantics/source-binding-summary.json')
ORDER BY byte_range[1];

function,byte_range,receiver_text,expected_callee_from_typed_fixture,wrong_typed_receiver_binding
phi_safe,"[199, 213]",value,std::option::Option::unwrap,True
phi_bad,"[320, 334]",value,std::option::Option::unwrap,True
mutated,"[445, 459]",value,std::option::Option::unwrap,True
shadowed,"[572, 586]",value,std::option::Option::unwrap,True
shadowed,"[604, 618]",value,std::option::Option::unwrap,True
shortcircuit,"[731, 745]",value,std::option::Option::unwrap,True
non_dominating,"[895, 909]",value,std::option::Option::unwrap,True
add_checked,"[946, 971]",x.checked_add(1),std::option::Option::unwrap,True
add_guarded,"[1021, 1046]",x.checked_add(1),std::option::Option::unwrap,True
helper,"[1136, 1150]",value,std::option::Option::unwrap,True


In [15]:
pg_mir_commands=[]
for pg_profile,pg_checks in [("checks_on","on"),("checks_off","off")]:
    pg_cmd=["rustup","run","nightly-2026-10-18","rustc","--crate-type","lib","--edition=2021","--crate-name","spur_grounding_semantics","--emit=mir","-Cpanic=unwind","-Cdebug-assertions=on",f"-Coverflow-checks={pg_checks}",str(pg_sem/"fixtures/lib.rs"),"-o",str(pg_sem/f"mir-{pg_profile}.mir")]
    pg_result=subprocess.run(pg_cmd,capture_output=True,text=True,timeout=30)
    pg_mir_commands.append({"profile":pg_profile,"command":pg_cmd,"exit_code":pg_result.returncode,"stderr":pg_result.stderr,"mir_sha256":pg_sha(pg_sem/f"mir-{pg_profile}.mir")})
    assert pg_result.returncode==0,pg_result.stderr
pg_probe.dump(pg_sem/"mir-commands.json",pg_mir_commands)
print(json.dumps(pg_mir_commands,indent=2))

[
  {
    "profile": "checks_on",
    "command": [
      "rustup",
      "run",
      "nightly-2026-10-18",
      "rustc",
      "--crate-type",
      "lib",
      "--edition=2021",
      "--crate-name",
      "spur_grounding_semantics",
      "--emit=mir",
      "-Cpanic=unwind",
      "-Cdebug-assertions=on",
      "-Coverflow-checks=on",
      "/Users/kevintruong/project/SpurCode/docs/research/artifacts/2026-11-01-panic-implementation-grounding/semantics/fixtures/lib.rs",
      "-o",
      "/Users/kevintruong/project/SpurCode/docs/research/artifacts/2026-11-01-panic-implementation-grounding/semantics/mir-checks_on.mir"
    ],
    "exit_code": 0,
    "stderr": "",
    "mir_sha256": "af9820110b34ff118a076d38bec0c078d0361e7604fda606d3fe8dbc26954f58"
  },
  {
    "profile": "checks_off",
    "command": [
      "rustup",
      "run",
      "nightly-2026-10-18",
      "rustc",
      "--crate-type",
      "lib",
      "--edition=2021",
      "--crate-name",
      "spur_grounding_semantics"

In [16]:
import mir_grounding as pg_mir
pg_mir_grounded=pg_mir.run()
print(json.dumps({k:v for k,v in pg_mir_grounded.items() if k!="rows"},indent=2))

{
  "passed": true,
  "profiles": 2,
  "typed_callee_comparisons": 22,
  "wrong_source_receiver_sites_confirmed_by_ordinary_mir": 11,
  "checks": [
    "mutated reaching assignment",
    "shadowed distinct locals",
    "short-circuit evaluation edge",
    "index assertion in both profiles",
    "overflow assertion only with checks on"
  ],
  "scope": "Version-pinned textual structural controls. Not general predicate extraction, CGA-instance correlation, or unbounded proof."
}


In [17]:
import normalization_ledger as pg_ledger
pg_ledger_result=pg_ledger.build()
pg_catalog_summary=json.loads((pg_sem/"catalog-verification.json").read_text())
pg_probe.dump(pg_out/"catalog-result-schemas.json",pg_conn.execute("DESCRIBE SELECT * FROM read_json_auto(?)",[str(pg_sem/"catalog-verification.json")]).fetchall())
print(json.dumps({"normalization_ledger":pg_ledger_result,"catalog_cases":pg_catalog_summary},indent=2))

{
  "normalization_ledger": {
    "functions": 16,
    "fact_fragments": 45,
    "constraint_bindings": 51,
    "automatic_translation": false
  },
  "catalog_cases": [
    {
      "name": "site_full_key",
      "note": "Two same-label same-body calls have distinct exact invocation spans.",
      "solve_id": "sol_a79f99cdf72e4661",
      "status": "sat"
    },
    {
      "name": "pair_key_collision",
      "note": "A caller/callee pair is insufficient as per-invocation identity.",
      "solve_id": "sol_09706b0ba0704976",
      "status": "unsat"
    },
    {
      "name": "nullable_unique_warning",
      "note": "Catalog unique is NULLS DISTINCT; uniqueness alone does not require identity completeness.",
      "solve_id": "sol_a6417bb6391b4de5",
      "status": "sat"
    },
    {
      "name": "missing_span_rejected",
      "note": "Explicit exact-tuple validation requires presence as well as agreement.",
      "solve_id": "sol_572da4a1b0234fbf",
      "status": "unsat"
    },
    {
 

In [18]:
import pandas as pd
pg_ext_loaded=pg_conn.execute("select extension_name,extension_version,loaded from duckdb_extensions() where extension_name in ('petgraph_ext','duckrouting','json','parquet') order by extension_name").fetchall()
pg_probe.dump(pg_out/"extensions-loaded.json",{"duckdb":duckdb.__version__,"extensions":pg_ext_loaded})
display(pd.DataFrame([{"function":r["function"],"from":r["source"],"to":r["target"],"path_exists":r["structural_path_exists"],"hops":r["hops"],"predicate":r["encoded_panic_status"],"panic_inputs":r["panic_inputs"]} for r in json.loads((pg_sem/"extension-paths.json").read_text())["rows"]]))
print("These graphs are source-authored control projections; both extensions agree with BFS, not with a claim of path feasibility.")

,function,from,to,path_exists,hops,predicate,panic_inputs
0,phi_safe,Entry,Panic,True,4.0,unsat,0
1,phi_safe,Panic,Entry,False,NaN,unsat,0
2,phi_bad,Entry,Panic,True,4.0,sat,255
3,phi_bad,Panic,Entry,False,NaN,sat,255


These graphs are source-authored control projections; both extensions agree with BFS, not with a claim of path feasibility.


In [19]:
SELECT name, status, note, solve_id
FROM read_json_auto('docs/research/artifacts/2026-11-01-panic-implementation-grounding/semantics/catalog-verification.json')
ORDER BY name;

name,status,note,solve_id
call_frontier_bound_1,unsat,Source-authored transition projection; bound 1 leaves Helper frontier. No global safety claim.,sol_f713fdc547844d50
call_frontier_bound_2,sat,Source-authored transition projection; bound 1 leaves Helper frontier. No global safety claim.,sol_c9c5dccc3fdd4608
forged_blob_rejected,unsat,Source binding cannot be replaced while retaining a certified site tuple.,sol_4cd66f45655942d1
missing_span_rejected,unsat,Explicit exact-tuple validation requires presence as well as agreement.,sol_572da4a1b0234fbf
nullable_unique_warning,sat,Catalog unique is NULLS DISTINCT; uniqueness alone does not require identity completeness.,sol_a6417bb6391b4de5
pair_key_collision,unsat,A caller/callee pair is insufficient as per-invocation identity.,sol_09706b0ba0704976
site_full_key,sat,Two same-label same-body calls have distinct exact invocation spans.,sol_a79f99cdf72e4661
site_owner_bound,sat,Author-grounded source/body/span tuple; no inferred compiler identity.,sol_f4df5b4bb40545ae
wrong_existing_body_rejected,unsat,A different existing same-file body must not own this site.,sol_4139927a024949fa


In [20]:
pg_grounding=pg_out/"grounding"
pg_checked=subprocess.run([sys.executable,str(pg_grounding/"verify.py"),"--current-source"],capture_output=True,text=True,timeout=30)
assert pg_checked.returncode==0,(pg_checked.stdout,pg_checked.stderr)
pg_worker_verified=json.loads(pg_checked.stdout)
pg_probe.dump(pg_out/"grounding-retained-verification.json",pg_worker_verified)
pg_fresh_dir=pg_grounding/"fresh-root-replay"
assert not pg_fresh_dir.exists(),"Use a new fresh-run output directory."
pg_grounding_command=[sys.executable,str(pg_grounding/"run.py"),"--out",str(pg_fresh_dir)]
pg_grounding_stdout=(pg_out/"root-grounding-run.stdout").open("w")
pg_grounding_stderr=(pg_out/"root-grounding-run.stderr").open("w")
pg_grounding_process=subprocess.Popen(pg_grounding_command,cwd=pg_repo,stdout=pg_grounding_stdout,stderr=pg_grounding_stderr)
pg_probe.dump(pg_out/"root-grounding-command.json",{"command":pg_grounding_command,"pid":pg_grounding_process.pid,"purpose":"Fresh notebook extraction/runtime/MIR/CGA control replay; no application build."})
print(json.dumps({"retained":pg_worker_verified,"fresh_replay_started":pg_grounding_process.pid},indent=2))

{
  "retained": {
    "cga_edge_kinds": {
      "BuiltinAssert": 2,
      "Called": 3,
      "FnPtrCast": 1,
      "VTableCalled": 1
    },
    "commands": 15,
    "counts": {
      "bodies": {
        "body_not_represented_as_graph_caller": 4,
        "expected_target_pair_present": 6
      },
      "compiler": {
        "separate_compiler_category": 5,
        "unresolved_target": 1
      },
      "lexical": {
        "expected_target_pair_present": 2,
        "missing_call_pair": 5
      },
      "receivers": {
        "expected_target_pair_present": 3,
        "unresolved_target": 4,
        "wrong_target_pair": 2
      }
    },
    "current_extraction_inputs_checked": 100,
    "files_hashed": 41,
    "hand_grounded_sites": 32,
    "runtime_inputs": 768,
    "runtime_panics": 640,
    "verified": true
  },
  "fresh_replay_started": 79680
}


In [21]:
pg_grounding_exit=pg_grounding_process.poll()
if pg_grounding_exit is None:
    pg_grounding_exit=pg_grounding_process.wait(timeout=20)
pg_grounding_stdout.close();pg_grounding_stderr.close()
assert pg_grounding_exit==0,(pg_out/"root-grounding-run.stderr").read_text()
pg_fresh_verify=subprocess.run([sys.executable,str(pg_grounding/"verify.py"),"--capture",str(pg_fresh_dir),"--current-source"],capture_output=True,text=True,timeout=30)
assert pg_fresh_verify.returncode==0,(pg_fresh_verify.stdout,pg_fresh_verify.stderr)
pg_fresh_verified=json.loads(pg_fresh_verify.stdout)
pg_probe.dump(pg_out/"grounding-fresh-verification.json",pg_fresh_verified)
pg_grounding_findings=json.loads((pg_fresh_dir/"findings.json").read_text())
print(json.dumps({"fresh_notebook_replay_exit":pg_grounding_exit,"verification":pg_fresh_verified,"compiler_uncertainty":pg_grounding_findings["compiler"]["macro_invocation_mapping"]},indent=2))

{
  "fresh_notebook_replay_exit": 0,
  "verification": {
    "cga_edge_kinds": {
      "BuiltinAssert": 2,
      "Called": 3,
      "FnPtrCast": 1,
      "VTableCalled": 1
    },
    "commands": 15,
    "counts": {
      "bodies": {
        "body_not_represented_as_graph_caller": 4,
        "expected_target_pair_present": 6
      },
      "compiler": {
        "separate_compiler_category": 5,
        "unresolved_target": 1
      },
      "lexical": {
        "expected_target_pair_present": 2,
        "missing_call_pair": 5
      },
      "receivers": {
        "expected_target_pair_present": 3,
        "unresolved_target": 4,
        "wrong_target_pair": 2
      }
    },
    "current_extraction_inputs_checked": 100,
    "files_hashed": 41,
    "hand_grounded_sites": 32,
    "runtime_inputs": 768,
    "runtime_panics": 640,
    "verified": true
  },
  "compiler_uncertainty": "UNKNOWN: raw Called edge points to macro definition expression; no expansion stack/invocation identity"
}


In [22]:
import importlib
pg_compiler_spec=importlib.util.spec_from_file_location("pg_compiler_probe",pg_sem/"compiler_probe.py")
pg_compiler_probe=importlib.util.module_from_spec(pg_compiler_spec)
pg_compiler_spec.loader.exec_module(pg_compiler_probe)
pg_compiler_models=pg_compiler_probe.verify()
pg_witness_binary=Path("/private/tmp/spur-panic-grounding-compiler-witness")
pg_witness_compile=["rustup","run","nightly-2026-10-18","rustc","--edition=2021","-Cpanic=unwind","-Cdebug-assertions=on","-Coverflow-checks=on",str(pg_sem/"compiler-witness-runtime.rs"),"-o",str(pg_witness_binary)]
pg_witness_compiled=subprocess.run(pg_witness_compile,capture_output=True,text=True,timeout=30)
assert pg_witness_compiled.returncode==0,pg_witness_compiled.stderr
pg_witness_replays=[]
for pg_row in pg_compiler_models["rows"]:
    if pg_row["status"]!="sat":
        continue
    pg_model=pg_row["witness"]
    pg_cmd=[str(pg_witness_binary),str(pg_model["x"]),str(pg_model["index"])]
    pg_run=subprocess.run(pg_cmd,capture_output=True,text=True,timeout=10)
    assert pg_run.returncode==0 and pg_run.stdout.startswith("panic\t"),(pg_row,pg_run.stdout,pg_run.stderr)
    pg_expected_reason={"compiler_any_panic":"assertion failed: index == 0","compiler_bounds_witness":"index out of bounds","compiler_overflow_witness":"attempt to add with overflow","compiler_assert_witness":"assertion failed: index == 0"}[pg_row["name"]]
    assert pg_expected_reason in pg_run.stdout,(pg_row,pg_run.stdout)
    pg_witness_replays.append({"case":pg_row["name"],"solve_id":pg_row["solve_id"],"model":pg_model,"command":pg_cmd,"exit_code":pg_run.returncode,"stdout":pg_run.stdout,"stderr":pg_run.stderr,"expected_reason":pg_expected_reason})
pg_probe.dump(pg_sem/"compiler-witness-replays.json",{"compile_command":pg_witness_compile,"compile_exit":pg_witness_compiled.returncode,"compile_stderr":pg_witness_compiled.stderr,"binary_sha256":pg_sha(pg_witness_binary),"source_sha256":pg_sha(pg_sem/"compiler-witness-runtime.rs"),"replays":pg_witness_replays})
pg_smt=importlib.reload(pg_smt)
pg_all_smt=pg_smt.run()
print(json.dumps({"compiler_models":pg_compiler_models,"witness_replays":pg_witness_replays,"all_fresh_formula_replays":pg_all_smt},indent=2))

{
  "compiler_models": {
    "passed": true,
    "cases": 5,
    "assignments": 3840,
    "runtime_inputs": 768,
    "runtime_panics": 640,
    "explicit_assumption_scope": "x:u8, index 0..=2, concrete object=(), overflow checks on, one entry call",
    "rows": [
      {
        "name": "compiler_any_panic",
        "status": "sat",
        "models": 640,
        "solve_id": "sol_9f1025ab3dc948b3",
        "core": null,
        "source_sha256": "d5b043f5f88a8a7373b0b9a87fa0e901dee74142532bd677131db429bcff24a0",
        "witness": {
          "index": 1,
          "x": 0
        },
        "retained_runtime_panic": true
      },
      {
        "name": "compiler_bounds_witness",
        "status": "sat",
        "models": 256,
        "solve_id": "sol_136a67312f0a45f8",
        "core": null,
        "source_sha256": "d5b043f5f88a8a7373b0b9a87fa0e901dee74142532bd677131db429bcff24a0",
        "witness": {
          "index": 2,
          "x": 0
        },
        "retained_runtime_panic": t

## Compiler lane checked in this notebook

The reviewed held-out controls were freshly replayed here: eight extraction/repeat runs, ordinary Rust execution/MIR, and the unchanged pinned CGA driver. All 100 extraction source/query inputs match the retained build inventory and current source. That freshness claim covers the extraction subsystem; two unrelated ACP adapter inputs differ from the old inventory.

Four lexical self-call sites are missing. Local closure shadowing and nested/module homonyms prevent a sound spelling-only repair. The receiver corpus has three expected pairs, four unresolved sites and two wrong string-length targets. Five targeted method syntax-site IDs survive repeat snapshots. Closure/async syntax does not imply the body ran: runtime counters confirm only two immediate method and two immediate free calls.

The successful CGA control retains concrete unit dispatch, an expanded call, two implicit assertions and an address cast. Address flow remains separate from execution. The expanded call points to the macro definition; invocation/expansion identity is still unknown. The known generic virtual substitution defect and full CLI capture remain outside this control.

For `object=()`, `x:u8`, `index=0..=2`, one entry call and overflow checks on, the explicit normalized predicate is `index>=2 OR x+x>255 OR index!=0`. All 768 outcomes match the runtime oracle (640 panics, 128 successful returns). Ordered solver witnesses are replayed again with their panic payloads: bounds at `(0,2)`, overflow at `(128,0)`, assertion at `(0,1)`. The five additional preflighted, persisted and reloaded models bring the total to 33 SOLVE/Z3 cases. This is a bounded concrete-receiver experiment, not automatic semantic extraction or arbitrary-trait safety.

In [23]:
pg_compiler_table=[{"case":r["case"],"x":r["model"]["x"],"index":r["model"]["index"],"actual_panic_payload":r["stdout"].strip().split("\t",1)[1],"solve_id":r["solve_id"]} for r in pg_witness_replays]
display(pd.DataFrame(pg_compiler_table))
print(json.dumps({"fresh_SOLVE_and_Z3_cases":pg_all_smt["executions"],"SAT":pg_all_smt["sat"],"UNSAT":pg_all_smt["unsat"],"primary_generic_assignments":pg_finite_result["assignments"]+pg_compiler_models["assignments"],"exhaustive_Rust_executions_in_final_runs":pg_runtime_result["executions"]+pg_fresh_verified["runtime_inputs"]},indent=2))

,case,x,index,actual_panic_payload,solve_id
0,compiler_any_panic,0,1,assertion failed: index == 0,sol_9f1025ab3dc948b3
1,compiler_bounds_witness,0,2,index out of bounds: the len is 2 but the inde...,sol_136a67312f0a45f8
2,compiler_overflow_witness,128,0,attempt to add with overflow,sol_d702f6040a6747e6
3,compiler_assert_witness,0,1,assertion failed: index == 0,sol_5fa01f88cdb34ffc


{
  "fresh_SOLVE_and_Z3_cases": 33,
  "SAT": 18,
  "UNSAT": 15,
  "primary_generic_assignments": 19712,
  "exhaustive_Rust_executions_in_final_runs": 8448
}


## Implementation decision and acceptance gates

Proceed with a narrow implementation in this order; no production implementation occurs in this notebook.

| Area | Grounded acceptance gate |
|---|---|
| Lexical self-call / receivers | Four recursion positives; closure-shadow and homonym negatives; remove incompatible receiver targets; retain unresolved cases when type authority is insufficient. |
| Bodies / calls / facts | Keep definition, snapshot body and each invocation identity distinct. Add free-call and closure/async execution identities. Validate presence, source/body/span ownership, typed origins, mutation/phi versions and evaluation guards. |
| Bounded predicates / replay | Record exact build/cfg, domain, assumptions and open frontiers. Replay SAT witnesses. Use UNSAT only for the complete declared scope; stale translations are rejected and missing facts remain inconclusive. |
| Compiler validation | Preserve raw instance/location evidence for traits, expansions and implicit checks. Keep address casts and uncertain virtual candidates separate. Require expansion stacks or explicit unknowns and separate regressions for the known CGA producer defect. |

The decisive blockers are target correctness and semantic translation authority. Hash binding alone cannot validate a translated formula. Algorithm agreement cannot validate a receiver or guard. The experiments support a SPUR-first candidate → source facts → bounded solve → replay pipeline with compiler validation where source evidence is incomplete.

Detailed evidence and replay: [research report](docs/research/artifacts/2026-11-01-panic-implementation-grounding/README.md).

In [24]:
pg_after={"protected":{p:pg_sha(pg_repo/p) for p in pg_protected_names},"graph_current_resolved":str(pg_current.resolve()),"graph_tables":[]}
assert pg_after["protected"]==pg_before["protected"],"Protected user/prior-notebook bytes changed"
assert pg_after["graph_current_resolved"]==pg_graph_protection["CURRENT"],"Graph CURRENT target changed"
for pg_input in pg_graph_protection["inputs"]:
    pg_hash=pg_sha(pg_input["path"])
    assert pg_hash==pg_input["sha256"],pg_input["path"]
    pg_after["graph_tables"].append({"path":pg_input["path"],"sha256":pg_hash})
pg_pins=json.loads((pg_fresh_dir/"provenance.json").read_text())["binary_pins"]
pg_after["cga_driver_sha256"]=pg_sha("/private/tmp/spur-cga-pilot-20261031/target/release/cga-driver")
pg_after["cga_launcher_sha256"]=pg_sha("/private/tmp/spur-cga-pilot-20261031/target/release/cargo-cga")
assert pg_after["cga_driver_sha256"]==pg_pins["driver"] and pg_after["cga_launcher_sha256"]==pg_pins["launcher"]
pg_after["passed"]=True
pg_probe.dump(pg_out/"protection-after.json",pg_after)
print(json.dumps({"passed":True,"protected_files":len(pg_after["protected"]),"published_graph_tables":len(pg_after["graph_tables"]),"original_CGA_binaries_unchanged":True},indent=2))

{
  "passed": true,
  "protected_files": 6,
  "published_graph_tables": 7,
  "original_CGA_binaries_unchanged": true
}


In [25]:
pg_frozen_check=subprocess.run([sys.executable,str(pg_out/"frozen_replay.py")],capture_output=True,text=True,timeout=30)
assert pg_frozen_check.returncode==0,(pg_frozen_check.stdout,pg_frozen_check.stderr)
pg_frozen_result=json.loads(pg_frozen_check.stdout)
pg_probe.dump(pg_out/"frozen-replay-result.json",pg_frozen_result)
assert {p:pg_sha(pg_repo/p) for p in pg_protected_names}==pg_before["protected"]
pg_final_verification={"passed":True,"frozen_replay":pg_frozen_result,"new_notebook":"2026-11-01-panic-implementation-grounding.ipynb","production_changes":False,"published_graph_unchanged":True,"original_CGA_binaries_unchanged":True,"source_semantics_automatically_extracted":False}
pg_probe.dump(pg_out/"final-notebook-verification.json",pg_final_verification)
print(json.dumps(pg_final_verification,indent=2))

{
  "passed": true,
  "frozen_replay": {
    "passed": true,
    "payload_hashes": 201,
    "primary_finite_assignments": 19712,
    "exhaustive_runtime_rows_verified": 8448,
    "solver_receipts_and_formula_provenance_verified": 33,
    "compiler_witness_payloads_verified": 4,
    "independent_grounding_captures_verified": 2,
    "extension_paths_verified": 4,
    "live_graph_or_notebook_read": false,
    "compiler_or_solver_reexecution": false
  },
  "new_notebook": "2026-11-01-panic-implementation-grounding.ipynb",
  "production_changes": false,
  "published_graph_unchanged": true,
  "original_CGA_binaries_unchanged": true,
  "source_semantics_automatically_extracted": false
}
